# Improved Modeling — LightGBM + Feature Engineering + Proper Validation

Three concrete improvements over `04_modeling.ipynb`:

| Problem | Fix |
|---|---|
| Missing non-linear / interaction features | **8 new derived features** (comorbidity count, age², age×conditions, etc.) |
| Test-set data leakage in early stopping | **Proper 85/15 train→val split** — test set is never seen during training |
| XGBoost only; no LightGBM | **LightGBM** (leaf-wise growth, faster convergence) + **weighted ensemble** |

All new features are derived from existing columns — **no new user inputs required in the app**.

Expected improvement: R² from 0.44 → ~0.50–0.54

In [70]:
# Install LightGBM if not already present
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'install', 'lightgbm', '-q'],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('WARNING: lightgbm install may have failed:', result.stderr[:300])

import pandas as pd
import numpy as np
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print(f'LightGBM : {lgb.__version__}')
import xgboost; print(f'XGBoost  : {xgboost.__version__}')

LightGBM : 4.6.0
XGBoost  : 3.1.3


## 1. Load Existing Train / Test Data

In [71]:
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()   # log1p(expenditure)
y_test  = pd.read_csv('y_test.csv').squeeze()

print(f'Train : {X_train.shape[0]:,} rows × {X_train.shape[1]} features')
print(f'Test  : {X_test.shape[0]:,} rows')
print(f'Target: log1p scale  mean={y_train.mean():.3f}  std={y_train.std():.3f}')
print(f'        dollar scale  mean=${np.expm1(y_train).mean():,.0f}  median=${np.expm1(y_train).median():,.0f}')

Train : 17,944 rows × 55 features
Test  : 4,487 rows
Target: log1p scale  mean=6.563  std=3.215
        dollar scale  mean=$7,627  median=$1,622


## 2. Add Derived Features

All 8 new features are computed from existing columns → no new user inputs in the app.

| Feature | Rationale |
|---|---|
| `num_conditions` | Comorbidity burden — strongest predictor after self-rated health |
| `is_elderly` | Age ≥65: Medicare eligibility triggers a spending discontinuity |
| `bmi_obese` | BMI ≥30: obesity independently raises spending 30–50% |
| `dual_eligible` | Medicare + Medicaid together: 2–3× per-capita spending of other Medicare enrollees |
| `cardiac_burden` | CHD + angina + MI + stroke: cardiovascular conditions are the leading cost driver |
| `age_sq` | Healthcare spending vs age is concave — quadratic term captures this |
| `age_x_conditions` | Older patients with more conditions have multiplicatively higher costs |
| `health_burden` | Poor self-rated health × many conditions → signals very high utilization |

In [72]:
DX_COLS = [
    'dx_hypertension', 'dx_coronary_heart_disease', 'dx_angina',
    'dx_myocardial_infarction', 'dx_stroke', 'dx_emphysema',
    'dx_high_cholesterol', 'dx_cancer', 'dx_arthritis',
    'dx_asthma', 'dx_adhd_add', 'dx_diabetes',
]
CARDIAC_COLS = [
    'dx_coronary_heart_disease', 'dx_angina',
    'dx_myocardial_infarction', 'dx_stroke',
]

def add_derived_features(X: pd.DataFrame) -> pd.DataFrame:
    """Add 8 derived features to the 55-column base matrix.
    All computable from existing columns — no new user input required.
    """
    X = X.copy()
    X['num_conditions']   = X[DX_COLS].sum(axis=1).astype(float)
    X['is_elderly']       = (X['age'] >= 65).astype(float)
    X['bmi_obese']        = (X['bmi'] >= 30).astype(float)
    X['dual_eligible']    = ((X['has_medicare'] == 1) & (X['has_medicaid'] == 1)).astype(float)
    X['cardiac_burden']   = X[CARDIAC_COLS].sum(axis=1).astype(float)
    X['age_sq']           = (X['age'] ** 2) / 1000.0          # normalised to ~0–7
    X['age_x_conditions'] = (X['age'] * X['num_conditions']) / 10.0
    X['health_burden']    = (X['self_rated_health'] * X['num_conditions']) / 10.0
    return X

X_train_ext = add_derived_features(X_train)
X_test_ext  = add_derived_features(X_test)
feature_cols_ext = list(X_train_ext.columns)
new_features = feature_cols_ext[55:]

print(f'Extended feature set: {len(feature_cols_ext)} columns ({len(new_features)} new)')
print(f'New features: {new_features}')
print()
print('New feature summary:')
print(X_train_ext[new_features].describe().T[['mean', 'std', 'min', 'max']].round(3).to_string())

Extended feature set: 63 columns (8 new)
New features: ['num_conditions', 'is_elderly', 'bmi_obese', 'dual_eligible', 'cardiac_burden', 'age_sq', 'age_x_conditions', 'health_burden']

New feature summary:
                   mean     std  min     max
num_conditions    1.351   1.676  0.0  11.000
is_elderly        0.236   0.425  0.0   1.000
bmi_obese         0.173   0.378  0.0   1.000
dual_eligible     0.056   0.230  0.0   1.000
cardiac_burden    0.138   0.489  0.0   4.000
age_sq            2.439   2.071  0.0   7.225
age_x_conditions  8.306  11.774  0.0  82.500
health_burden     0.381   0.567  0.0   5.500


## 3. Fix Data Leakage — Proper Validation Split

The original `04_modeling.ipynb` passed `eval_set=[(X_test, y_test)]` to XGBoost, allowing
the held-out test set to influence training via early stopping. Fixed below:
carve 15 % of **train** into a validation set; the test set is never touched until final evaluation.

In [73]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_ext, y_train, test_size=0.15, random_state=42)

print(f'Training   : {X_tr.shape[0]:,} rows  ({X_tr.shape[0]/len(X_train_ext)*100:.0f}%)')
print(f'Validation : {X_val.shape[0]:,} rows  (early stopping only; never evaluated as final test)')
print(f'Test       : {X_test_ext.shape[0]:,} rows  (final evaluation — untouched during training)')

Training   : 15,252 rows  (85%)
Validation : 2,692 rows  (early stopping only; never evaluated as final test)
Test       : 4,487 rows  (final evaluation — untouched during training)


## 4. Evaluation Helper

In [74]:
def evaluate(name, y_true, y_pred_log):
    """Metrics on both log1p and dollar scales."""
    r2       = r2_score(y_true, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_true, y_pred_log))
    mae_log  = mean_absolute_error(y_true, y_pred_log)
    y_true_d = np.expm1(y_true)
    y_pred_d = np.maximum(np.expm1(y_pred_log), 0)
    rmse_d   = np.sqrt(mean_squared_error(y_true_d, y_pred_d))
    mae_d    = mean_absolute_error(y_true_d, y_pred_d)
    pred_mean = y_pred_d.mean()
    return dict(model=name, r2=round(float(r2),4),
                rmse_log=round(float(rmse_log),4), mae_log=round(float(mae_log),4),
                rmse_dollar=round(float(rmse_d),2), mae_dollar=round(float(mae_d),2),
                pred_mean=round(float(pred_mean),2))

results = []
actual_mean = float(np.expm1(y_test).mean())
print(f'Test set actual mean spending: ${actual_mean:,.0f}')

Test set actual mean spending: $7,690


## 5. LightGBM

LightGBM uses **leaf-wise** tree growth (vs XGBoost's depth-wise), which typically achieves
better accuracy in fewer iterations on tabular data. Key settings:
- `num_leaves=127`: controls model complexity (equivalent to depth≈7, but more flexible)
- `learning_rate=0.02` + `n_estimators=3000`: slower learning with more trees for better generalisation
- Early stopping on validation set (not test set)

In [75]:
print('Training LightGBM...')

lgb_model = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',
    n_estimators=3000,
    learning_rate=0.02,
    num_leaves=127,
    min_child_samples=20,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)

callbacks = [
    lgb.early_stopping(stopping_rounds=75, verbose=False),
    lgb.log_evaluation(period=-1),
]

lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=callbacks,
)

print(f'Best iteration : {lgb_model.best_iteration_}')

y_pred_lgb = lgb_model.predict(X_test_ext)
res_lgb = evaluate('LightGBM (63 feat)', y_test, y_pred_lgb)
results.append(res_lgb)

print(f"  R²          : {res_lgb['r2']:.4f}")
print(f"  RMSE (log)  : {res_lgb['rmse_log']:.4f}")
print(f"  MAE  ($)    : ${res_lgb['mae_dollar']:,.0f}")
print(f"  Pred mean   : ${res_lgb['pred_mean']:,.0f}  (actual ${actual_mean:,.0f})")

Training LightGBM...
Best iteration : 254
  R²          : 0.4365
  RMSE (log)  : 2.4115
  MAE  ($)    : $6,176
  Pred mean   : $2,728  (actual $7,690)


## 6. XGBoost v2 — Improved Hyperparameters + Proper Early Stopping

In [76]:
print('Training XGBoost v2 (63 features, proper validation, improved hyperparameters)...')

xgb_v2 = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.7,
    colsample_bylevel=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    gamma=0.05,
    n_jobs=-1,
    random_state=42,
    verbosity=0,
    early_stopping_rounds=75,   # must be in constructor in XGBoost >= 2.0
)

xgb_v2.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

print(f'Best iteration : {xgb_v2.best_iteration}')

y_pred_xgb2 = xgb_v2.predict(X_test_ext)
res_xgb2 = evaluate('XGBoost v2 (63 feat)', y_test, y_pred_xgb2)
results.append(res_xgb2)

print(f"  R²          : {res_xgb2['r2']:.4f}")
print(f"  RMSE (log)  : {res_xgb2['rmse_log']:.4f}")
print(f"  MAE  ($)    : ${res_xgb2['mae_dollar']:,.0f}")
print(f"  Pred mean   : ${res_xgb2['pred_mean']:,.0f}  (actual ${actual_mean:,.0f})")

Training XGBoost v2 (63 features, proper validation, improved hyperparameters)...
Best iteration : 670
  R²          : 0.4437
  RMSE (log)  : 2.3961
  MAE  ($)    : $6,163
  Pred mean   : $2,906  (actual $7,690)


## 7. Weighted Ensemble — LightGBM + XGBoost v2

Grid-search over the blend weight on the **validation set** (not the test set) to avoid leakage.
Test set is only used to report final performance.

In [77]:
# ── Find optimal weight on validation set (not test) ──────────────────────────
y_val_pred_lgb  = lgb_model.predict(X_val)
y_val_pred_xgb2 = xgb_v2.predict(X_val)

best_val_r2 = -np.inf
best_w = 0.5
for w in np.arange(0.0, 1.01, 0.05):
    y_val_ens = w * y_val_pred_lgb + (1 - w) * y_val_pred_xgb2
    val_r2 = r2_score(y_val, y_val_ens)
    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        best_w = w

lgb_weight = round(best_w, 2)
xgb_weight = round(1 - best_w, 2)
print(f'Optimal weights (by validation R²): LGB={lgb_weight:.2f}  XGB={xgb_weight:.2f}')

# ── Evaluate on test set ───────────────────────────────────────────────────────
y_pred_ens = lgb_weight * y_pred_lgb + xgb_weight * y_pred_xgb2
res_ens = evaluate('Ensemble (LGB+XGB)', y_test, y_pred_ens)
results.append(res_ens)

print(f"  R²          : {res_ens['r2']:.4f}")
print(f"  RMSE (log)  : {res_ens['rmse_log']:.4f}")
print(f"  MAE  ($)    : ${res_ens['mae_dollar']:,.0f}")
print(f"  Pred mean   : ${res_ens['pred_mean']:,.0f}  (actual ${actual_mean:,.0f})")

Optimal weights (by validation R²): LGB=0.15  XGB=0.85
  R²          : 0.4440
  RMSE (log)  : 2.3954
  MAE  ($)    : $6,160
  Pred mean   : $2,869  (actual $7,690)


## 8. Full Comparison — Old vs New Models

In [78]:
# ── Load original XGBoost for baseline comparison ─────────────────────────────
xgb_orig = joblib.load(os.path.join('..', 'models', 'xgboost.pkl'))
y_pred_xgb_orig = xgb_orig.predict(X_test)   # 55-feature version
res_orig = evaluate('XGBoost (original, 55 feat)', y_test, y_pred_xgb_orig)

print('=' * 78)
print('FULL MODEL COMPARISON'.center(78))
print('=' * 78)
print(f'{"Model":<35} {"R²":>8} {"RMSE_log":>10} {"MAE ($)":>10} {"Pred Mean":>11}')
print('-' * 78)

all_results = [res_orig] + results
for r in all_results:
    print(f"  {r['model']:<33} {r['r2']:>8.4f} {r['rmse_log']:>10.4f}"
          f" ${r['mae_dollar']:>9,.0f} ${r['pred_mean']:>9,.0f}")

print('=' * 78)
print(f'  Actual test mean: ${actual_mean:,.0f}')

best = max(results, key=lambda x: x['r2'])
delta_r2 = best['r2'] - res_orig['r2']
delta_mae = res_orig['mae_dollar'] - best['mae_dollar']
print(f'\n  Best new model : {best["model"]}')
print(f'  R² improvement : +{delta_r2:.4f}  ({delta_r2/res_orig["r2"]*100:.1f}% relative gain)')
print(f'  MAE improvement: -${delta_mae:,.0f} per person')

                            FULL MODEL COMPARISON                             
Model                                     R²   RMSE_log    MAE ($)   Pred Mean
------------------------------------------------------------------------------
  XGBoost (original, 55 feat)         0.4398     2.4044 $    6,220 $    3,089
  LightGBM (63 feat)                  0.4365     2.4115 $    6,176 $    2,728
  XGBoost v2 (63 feat)                0.4437     2.3961 $    6,163 $    2,906
  Ensemble (LGB+XGB)                  0.4440     2.3954 $    6,160 $    2,869
  Actual test mean: $7,690

  Best new model : Ensemble (LGB+XGB)
  R² improvement : +0.0042  (1.0% relative gain)
  MAE improvement: -$60 per person


## 9. Feature Importance — LightGBM

In [79]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

importances = lgb_model.feature_importances_
idx = np.argsort(importances)[-25:]   # top 25

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(np.array(feature_cols_ext)[idx], importances[idx], color='steelblue')
ax.set_title('LightGBM — Top 25 Feature Importances (gain)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance (gain)')
ax.tick_params(axis='y', labelsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join('..', 'models', 'feature_importance_lgb.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: models/feature_importance_lgb.png')

# Print new features' importance ranks
feat_imp = dict(zip(feature_cols_ext, importances))
print('\nNew feature importances (rank / total):')
all_sorted = sorted(feat_imp.items(), key=lambda x: x[1], reverse=True)
name_to_rank = {name: rank+1 for rank, (name, _) in enumerate(all_sorted)}
for f in new_features:
    print(f"  {f:<25} rank {name_to_rank[f]:>2} / {len(feature_cols_ext)}  importance={feat_imp[f]:.1f}")

Saved: models/feature_importance_lgb.png

New feature importances (rank / total):
  num_conditions            rank 22 / 63  importance=309.0
  is_elderly                rank 62 / 63  importance=4.0
  bmi_obese                 rank 50 / 63  importance=29.0
  dual_eligible             rank 41 / 63  importance=79.0
  cardiac_burden            rank 51 / 63  importance=29.0
  age_sq                    rank 15 / 63  importance=586.0
  age_x_conditions          rank  7 / 63  importance=1519.0
  health_burden             rank 13 / 63  importance=706.0


## 10. Save Models + Update JSON Artifacts

In [80]:
models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)

# ── Save new model files ───────────────────────────────────────────────────────
joblib.dump(lgb_model, os.path.join(models_dir, 'lightgbm.pkl'))
joblib.dump(xgb_v2,   os.path.join(models_dir, 'xgboost_v2.pkl'))

# ── Update model_meta.json ────────────────────────────────────────────────────
meta_path = os.path.join(models_dir, 'model_meta.json')
with open(meta_path) as f:
    meta = json.load(f)

meta['models']['LightGBM'] = {
    'file': 'lightgbm.pkl',
    'needs_scaling': False,
    'target_space': 'log1p',
    'smearing_factor': None,
    'feature_set': 'extended',
}
meta['models']['XGBoost v2'] = {
    'file': 'xgboost_v2.pkl',
    'needs_scaling': False,
    'target_space': 'log1p',
    'smearing_factor': None,
    'feature_set': 'extended',
}
meta['best_model'] = 'Ensemble (LGB+XGB)'
meta['ensemble'] = {
    'models': ['LightGBM', 'XGBoost v2'],
    'weights': [lgb_weight, xgb_weight],
    'feature_set': 'extended',
    'note': 'Weights tuned on validation split (15% of train), not test set.',
}
meta['feature_cols_extended'] = feature_cols_ext

with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print('Updated: models/model_meta.json')

# ── Update model_results.json ─────────────────────────────────────────────────
results_path = os.path.join(models_dir, 'model_results.json')
with open(results_path) as f:
    orig_results = json.load(f)

# Remove previous LGB/ensemble entries if re-running
orig_results = [r for r in orig_results
                if r['model'] not in ('LightGBM (63 feat)', 'XGBoost v2 (63 feat)', 'Ensemble (LGB+XGB)')]
orig_results.extend(results)

with open(results_path, 'w') as f:
    json.dump(orig_results, f, indent=2)

print('Updated: models/model_results.json')

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print('Files saved to models/:')
print('  lightgbm.pkl')
print('  xgboost_v2.pkl')
print('  feature_importance_lgb.png')
print('  model_meta.json  (updated: ensemble weights, feature_cols_extended)')
print('  model_results.json  (updated: LGB + XGB v2 + ensemble results)')
print()
print(f'Ensemble weights: {lgb_weight:.2f} × LightGBM  +  {xgb_weight:.2f} × XGBoost v2')
print(f'Primary model R²: {res_ens["r2"]:.4f}  (was {res_orig["r2"]:.4f} with original XGBoost)')
print(f'Primary model MAE: ${res_ens["mae_dollar"]:,.0f}  (was ${res_orig["mae_dollar"]:,.0f})')

Updated: models/model_meta.json
Updated: models/model_results.json

Files saved to models/:
  lightgbm.pkl
  xgboost_v2.pkl
  feature_importance_lgb.png
  model_meta.json  (updated: ensemble weights, feature_cols_extended)
  model_results.json  (updated: LGB + XGB v2 + ensemble results)

Ensemble weights: 0.15 × LightGBM  +  0.85 × XGBoost v2
Primary model R²: 0.4440  (was 0.4398 with original XGBoost)
Primary model MAE: $6,160  (was $6,220)
